[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/07_soil-data/07_01_soil_data_ssurgo.ipynb)


# Module 7: Soil Data for H&H Modeling with NRCS SSURGO
## Accessing and Processing Official Soil Survey Data Through a Web API

### Welcome!
This module teaches you how to pull official NRCS soil survey data (SSURGO) directly into Python, so you can extract the soil properties that matter for hydrologic modeling: hydrologic soil group, saturated hydraulic conductivity (Ksat), soil texture, and available water capacity. Along the way, this module also teaches something more general: how a web API actually works, using raw requests instead of a wrapper library. By the end, you will be able to query soil data for a single point or for an entire watershed, and export engineering-ready tables.

### What You'll Work Through Today:
- Understand how SSURGO organizes soil data into map units, components, and horizons
- Understand what a web API request and response actually look like
- Query soil properties at a single point using the NRCS Soil Data Access (SDA) API
- Query soil properties across a full watershed and aggregate them correctly
- Visualize hydrologic soil groups across a watershed
- Export model-ready soil summary tables

### Module Structure:
1. **Mental Models** - SSURGO as a filing cabinet of soil surveys
2. **How a Web API Works** - Requests, responses, and JSON
3. **The Data Source** - NRCS Soil Data Access (SDA)
4. **Workspace Setup** - Installing libraries and loading the watershed
5. **Point Query** - Soil properties at a single location
6. **Watershed Query** - Soil properties across an entire watershed
7. **Visualization** - Mapping hydrologic soil groups
8. **Export** - Model-ready output tables
9. **Beyond the API** - SSURGO Portal and gNATSGO

### Prerequisites
This module assumes you have completed Module 1 (Python fundamentals) and Module 3 (vector data, coordinate reference systems, GeoPandas). Module 5 (API-based data retrieval) is helpful background but not required; this module explains API concepts from scratch.


## Using AI in This Module

This module writes raw web requests by hand instead of using a wrapper library, so the exact syntax of a query or a request payload may feel unfamiliar at first. Use your AI assistant the same way you have in every other module:

- *"Explain what this SQL query is asking for, in plain English."*
- *"My SDA query returned an empty table. What are the most likely reasons?"*
- *"I got a 400 error back from the API. Here is my query and the error message. What is wrong?"*

The SDA service occasionally changes response formatting or has brief outages. If a live query fails during class or self-study, this notebook includes saved fallback data in the `data/` folder so you can keep working through the lesson without losing your place. As always, review what the AI suggests before you run it. You are responsible for the query, not the assistant.


## Part 1: Mental Models - SSURGO as a Filing Cabinet 🗄️

### Think of SSURGO as a National Filing Cabinet of Soil Surveys

The Soil Survey Geographic Database (SSURGO) is the official soil survey of the United States, produced by the USDA Natural Resources Conservation Service (NRCS). It is organized in three nested levels, and the analogy of a filing cabinet makes the structure easy to hold in your head.

- **Map unit** = a drawer. This is an area drawn on the map, identified by a `mukey` (map unit key). Map units are the polygons you would see if you opened Web Soil Survey and looked at a map.
- **Component** = a folder inside the drawer. A single map unit is rarely one uniform soil. It is usually a mix of soil types, called components, each identified by a `cokey` (component key). Each component has a `comppct_r` value: the percentage of the map unit's area it represents.
- **Horizon** = a page inside the folder. Each component is a vertical soil profile, and it is broken into depth layers called horizons, identified by a `chkey` (horizon key). A horizon has a top depth (`hzdept_r`) and bottom depth (`hzdepb_r`), and this is where properties like Ksat and available water capacity are actually stored.

### The One-to-Many Warning

This structure means a single point on the map does not correspond to a single soil. A point falls inside exactly one map unit, but that map unit can contain several components, and each component can contain several horizons. Later in this module you will see this play out with real data: querying "the soil at a point" will return several rows, not one, and you will need to decide how to combine them into a single number. Any time you report a SSURGO-derived value, you should be able to say whether it is the dominant component only, a percentage-weighted average across components, a depth-weighted average across horizons, or some combination. This module builds that habit from the start.

### Key Terms

| Term | What It Is | Units / Example |
|---|---|---|
| `mukey` | Map unit key, identifies a mapped soil area | e.g. `104963` |
| `musym` / `muname` | Map unit symbol and name | e.g. "162", "Poposhia-Trimad complex" |
| `cokey` | Component key, identifies one soil type within a map unit | e.g. `27266259` |
| `compname` | Component name | e.g. "Poposhia" |
| `comppct_r` | Representative percentage of the map unit this component covers | percent (0-100) |
| `chkey` | Horizon key, identifies one depth layer within a component | e.g. `81398856` |
| `hzdept_r` / `hzdepb_r` | Horizon top and bottom depth | centimeters |
| `ksat_r` | Representative saturated hydraulic conductivity | micrometers per second (µm/s) |
| `hydgrp` | Hydrologic soil group (component level) | A, B, C, D, or dual classes like C/D |
| `awc_r` | Available water capacity | fraction of soil volume, e.g. 0.20 |
| `drainagecl` | Drainage class | e.g. "Well drained" |

### The Engineering Question This Module Answers

*"What are the infiltration and runoff characteristics of the soils in my watershed?"* This single question feeds several standard H&H methods:

- **Hydrologic soil group** drives curve number selection (Module 4 covered curve numbers from land cover; SSURGO is where the soil half of that calculation comes from).
- **Ksat** is a starting point for Green-Ampt infiltration parameters.
- **Available water capacity** informs soil moisture storage and deficit-based methods.

### SSURGO vs gSSURGO vs gNATSGO, in One Table

| Product | What It Is |
|---|---|
| SSURGO | The original vector soil survey: map unit polygons plus linked tabular data, organized by individual survey area |
| gSSURGO | SSURGO repackaged into a seamless national raster and geodatabase, same underlying data, easier large-area access |
| gNATSGO | A composite product that fills SSURGO's gaps with STATSGO2 (coarser, older survey) and Raster Soil Survey data, for full national coverage |

This module works directly with SSURGO through NRCS's own web service. gNATSGO is introduced again briefly at the end, as an alternative for large-area raster work.


## Part 2: How a Web API Works 🔌

### The Records-Office Analogy

Every time you have used `pip install` or opened a web page, a computer somewhere sent a request to another computer and got a response back. An API (Application Programming Interface) is simply an agreed-upon way of asking for data over the internet.

Think of it like a records office with a counter. You do not walk into the back room and grab a file yourself. Instead:

1. You fill out a **request form**: which records you want, in what format.
2. You hand the form to the clerk at the counter. In web terms, this is sending an HTTP request to a specific **endpoint** (a URL).
3. The clerk looks up exactly what you asked for and hands back a **response**: a structured package of data, usually as JSON (a text format for nested lists and key-value pairs, very similar to a Python dictionary).

### GET vs. POST

There are two common ways to send a request:

- **GET** requests ask for something using a short URL, often with parameters attached at the end. Good for simple lookups.
- **POST** requests send a larger payload in the body of the request, not the URL. This is used when the request itself is complex, such as an entire SQL query.

The Soil Data Access service you will use in this module accepts SQL queries, which can be long and detailed, so it uses POST.

### Status Codes

The response always comes back with a status code, a three-digit number telling you what happened:

- **200** means success, here is your data.
- **400** means your request was malformed (the server did not understand what you asked for).
- **500** means something broke on the server's side.

Checking the status code before trying to use the response is a basic but important habit. The code cell below sends a very small, harmless request and prints exactly what comes back, so you can see a request and response with your own eyes before anything else happens in this module.


In [ ]:
# A minimal request/response round trip
#
# We are not asking for soil data yet. This cell only demonstrates
# what a request and a response actually look like.
import requests

# This is the endpoint: the URL of NRCS's Soil Data Access web service
sda_url = "https://sdmdataaccess.sc.egov.usda.gov/Tabular/post.rest"

# This is the request payload
#
# "query" holds a plain SQL statement
# "format" tells the server how to shape the response
demo_query = "SELECT TOP 3 mukey, musym, muname FROM mapunit"
demo_payload = {
    "query": demo_query,
    "format": "JSON+COLUMNNAME",
}

# Send the request as a POST, because we are sending a query, not just a URL
response = requests.post(sda_url, json=demo_payload)

# Print the status code first
#
# 200 means the server understood the request and is sending data back
print(f"Status code: {response.status_code}")

# Print the raw response text
#
# This is JSON: a text format for structured data.
# Notice it looks almost exactly like a Python dictionary of lists.
print("Raw response:")
print(response.text)


### What Just Happened

The response came back as a dictionary with one key, `"Table"`, containing a list of rows. The first row is the column names (because we asked for `JSON+COLUMNNAME` format), and every row after that is one record. This raw shape is not convenient to work with directly, so the rest of this module wraps that same pattern into a pandas DataFrame right after every request.

🤖 **Try asking your AI assistant:** *"Explain the difference between a GET request and a POST request, and why sending a SQL query to a web API would normally use POST instead of GET."*


## Part 3: The Data Source - NRCS Soil Data Access (SDA) 🌐

### What SDA Is

Soil Data Access is a set of NRCS web services built specifically for requesting soil survey data without downloading an entire state or county database. According to NRCS, SDA exists to support exactly the kind of request this module needs:

- An ad hoc area of interest, defined by you, not by a fixed survey boundary
- Real-time access, with no local database to install or maintain
- Selected tabular and spatial attributes, not the full SSURGO schema
- Results bundled by your request, not by the underlying survey area organization

This is the right tool for a quick, project-specific soil query. If you instead need a full local, offline database with map unit rasters and Soil Data Viewer outputs, NRCS provides a separate desktop tool called SSURGO Portal, covered conceptually at the end of this module.

### Today's Engineering Scenario

You are starting a screening-level hydrologic model of a HUC-12 watershed in Wyoming (the same watershed used in Module 3). Field investigation is not budgeted for this stage of the project. Before building the model, you need two things: the hydrologic soil group for curve number selection, and Ksat as a defensible starting point for infiltration parameters. Both are available through the official soil survey, and SDA lets you pull them directly into your workflow.

### A Note on Limitations

SSURGO is survey-scale mapping, typically compiled at a scale around 1:24,000. The values you retrieve are survey estimates for a mapped soil type, not a laboratory measurement of your specific site. Treat everything in this module as a screening-level input, useful for initial model setup and comparison across a watershed, not a substitute for site investigation on a design-level project.


## Part 4: Workspace Setup 🛠️

### Installing and Importing Libraries

Google Colab already includes `requests`, `pandas`, `geopandas`, `shapely`, and `matplotlib`, so no installation is needed. The cell below just imports everything this module uses, grouped by purpose.


In [ ]:
# Web requests
import requests

# Tabular data
import pandas as pd

# Spatial data
import geopandas as gpd
import shapely.ops

# Plotting
import matplotlib.pyplot as plt

# The Soil Data Access tabular endpoint used throughout this module
SDA_URL = "https://sdmdataaccess.sc.egov.usda.gov/Tabular/post.rest"

print("Libraries imported. Ready to query SSURGO data.")


### A Small Reusable Function for Querying SDA

Every query in this module follows the same three steps: build a SQL string, POST it, turn the JSON response into a DataFrame. Instead of repeating those three steps by hand each time, this cell wraps them into one small function. This is introduced now, after you have already seen the raw request/response pattern once in Part 2, so the function is not hiding anything new.


In [ ]:
def query_sda(sql):
    """Send a SQL query to Soil Data Access and return the result as a DataFrame.

    Parameters
    ----------
    sql : str
        A T-SQL query written against the SSURGO/SDA tables and functions.

    Returns
    -------
    pandas.DataFrame
        The query result. Returns an empty DataFrame if SDA has no matching rows.
    """
    payload = {"query": sql, "format": "JSON+COLUMNNAME"}
    response = requests.post(SDA_URL, json=payload, timeout=30)

    # Always check the status code before trying to use the response
    if response.status_code != 200:
        print(f"Request failed with status code {response.status_code}")
        print(response.text[:500])
        return pd.DataFrame()

    result = response.json()

    # An area with no SSURGO coverage returns an empty dictionary, not an error
    if "Table" not in result:
        print("No data returned. The query may be outside SSURGO's mapped area.")
        return pd.DataFrame()

    table = result["Table"]
    columns = table[0]
    rows = table[1:]
    return pd.DataFrame(rows, columns=columns)


# Quick check: this should return the same three rows we saw printed in Part 2
test_df = query_sda("SELECT TOP 3 mukey, musym, muname FROM mapunit")
print(test_df)


### Loading the Study Watershed

This module reuses the same Wyoming HUC-12 watershed from Module 3: watershed `101900090108`, "Town of South Greeley." If you completed Module 3, this file will look familiar. Upload the same ZIP file you used there.


In [ ]:
from google.colab import files

print("Please upload this file:")
print("NHD__Watershed_Boundaries_HUC_12_Selected.zip")
print()
print("Click 'Choose Files' below and select it.")

uploaded = files.upload()

print(f"\nUploaded {len(uploaded)} file(s):")
for filename in uploaded.keys():
    print(f"   - {filename}")


In [ ]:
# Read the watershed shapefile directly from the ZIP archive
#
# "zip://" tells GeoPandas the shapefile is stored inside a ZIP file
watersheds = gpd.read_file(
    'zip://NHD__Watershed_Boundaries_HUC_12_Selected.zip'
)

print(f"Loaded {len(watersheds)} watershed polygons")
print(f"Original CRS: {watersheds.crs}")

# Select our target watershed by its HUC-12 code
target_huc = '101900090108'
target_watershed = watersheds[watersheds['HUC12'] == target_huc]

print(f"\nTarget watershed: {target_watershed['Name'].iloc[0]}")
print(f"Area: {target_watershed['AreaAcres'].iloc[0]:,.0f} acres")

# SDA expects coordinates in WGS84 (EPSG:4326), the standard latitude/longitude system
#
# Our watershed file uses a different, projected CRS, so we reproject it here
target_wgs84 = target_watershed.to_crs(4326)

print(f"Reprojected CRS: {target_wgs84.crs}")


## Part 5: Point Query - Soil Properties at a Single Location 📍

### The Engineering Task

Given one location, latitude and longitude, find the SSURGO map unit at that point, then retrieve the soil properties that matter for hydrology: hydrologic soil group, Ksat, texture, and available water capacity.

### Step 1: Find the Map Unit at a Point

SDA provides a spatial helper function that takes a point written in WKT (Well-Known Text) format and returns the `mukey` it falls inside. WKT writes a point as `POINT(longitude latitude)`. Notice the order: longitude first, then latitude. This is the opposite of how people normally say coordinates out loud ("41 north, 105 west"), and it is a common source of bugs. Get this order backwards and your point lands in the wrong hemisphere entirely.

We will use the centroid of our target watershed as the example point.


In [ ]:
# Get a representative point inside the watershed
#
# .centroid finds the geometric center of the polygon
point_geom = target_wgs84.geometry.iloc[0].centroid

point_lon = point_geom.x
point_lat = point_geom.y

print(f"Point location: longitude={point_lon:.6f}, latitude={point_lat:.6f}")

# Build the WKT string
#
# Remember: WKT point order is (longitude latitude), not (latitude longitude)
point_wkt = f"POINT({point_lon} {point_lat})"
print(f"WKT: {point_wkt}")

# Ask SDA which map unit contains this point
mukey_sql = f"""
SELECT mukey
FROM SDA_Get_Mukey_from_intersection_with_WktWgs84('{point_wkt}')
"""

point_mukey_df = query_sda(mukey_sql)
print(point_mukey_df)

point_mukey = point_mukey_df['mukey'].iloc[0]
print(f"\nThe map unit at this point is mukey {point_mukey}")


### Step 2: Retrieve Component and Horizon Data for This Map Unit

Now that we have a `mukey`, we join across the three levels described in Part 1: map unit, component, and horizon. Each row in the result is one horizon, tagged with which component and which map unit it belongs to.


In [ ]:
point_detail_sql = f"""
SELECT
    mu.mukey,
    mu.musym,
    mu.muname,
    co.cokey,
    co.compname,
    co.comppct_r,
    co.hydgrp,
    co.drainagecl,
    ch.chkey,
    ch.hzdept_r,
    ch.hzdepb_r,
    ch.ksat_r,
    ch.awc_r
FROM mapunit mu
INNER JOIN component co ON mu.mukey = co.mukey
INNER JOIN chorizon ch ON co.cokey = ch.cokey
WHERE mu.mukey = {point_mukey}
ORDER BY co.comppct_r DESC, ch.hzdept_r ASC
"""

point_detail = query_sda(point_detail_sql)

# The API returns every value as text, so convert the numeric columns
numeric_cols = ['comppct_r', 'hzdept_r', 'hzdepb_r', 'ksat_r', 'awc_r']
for col in numeric_cols:
    point_detail[col] = pd.to_numeric(point_detail[col], errors='coerce')

print(f"Retrieved {len(point_detail)} horizon records")
point_detail


### What This Table Tells Us

Notice this single map unit already contains more than one component, and each component has multiple horizons. This is the one-to-many structure from Part 1, visible in real data.

Look at the `comppct_r` column: the components shown here may not add up to 100 percent. SSURGO map units often include minor components, such as rock outcrop or a soil type with no depth data on file, that carry a percentage but have no horizon records. Because the query above uses an `INNER JOIN` against the horizon table, any component without horizon data is silently dropped from the result. This is not a bug in the query. It is the nature of an inner join: it only keeps rows that match on both sides.

The engineering implication: if you later average `ksat_r` weighted by `comppct_r`, you should weight by the percentage of components actually present in your result, not assume the percentages sum to 100. Part 6 handles this explicitly.

### Step 3: Convert Ksat to Engineering Units

SSURGO stores Ksat in micrometers per second (µm/s), which is not the unit most H&H engineers think in day to day. The conversion to inches per hour is straightforward:

1 µm/s = 0.14173 in/hr


In [ ]:
# Convert Ksat from micrometers per second to inches per hour
point_detail['ksat_in_hr'] = point_detail['ksat_r'] * 0.14173

# Convert Ksat from micrometers per second to centimeters per hour
point_detail['ksat_cm_hr'] = point_detail['ksat_r'] * 0.36

print(point_detail[['compname', 'comppct_r', 'hzdept_r', 'hzdepb_r', 'ksat_r', 'ksat_in_hr']])


🤖 **Try asking your AI assistant:** *"Explain how mukey, cokey, and chkey connect SSURGO map units, components, and horizons. Use a simple watershed hydrology example."*

### Engineering Caution: Ksat Is a Starting Point, Not a Calibrated Parameter

The Ksat values you just retrieved are useful as a first estimate for infiltration modeling (Green-Ampt, or as a sanity check on curve number results), but they should not be treated as a calibrated final parameter. In practice, Ksat is adjusted during model calibration against observed streamflow or infiltration testing. Use these SSURGO values to get a defensible starting point, then confirm or adjust with calibration whenever observed data is available.


## Part 6: Watershed Query - Soil Properties Across an Entire Watershed 🗺️

### The Engineering Task

A single point tells you about one location. For model parameterization, you usually need to summarize soil properties across an entire watershed: which hydrologic soil group dominates, and what a reasonable watershed-average Ksat looks like.

### Step 1: Prepare the Watershed Boundary as WKT

The same spatial helper function from Part 5 also accepts a polygon, not just a point. But a watershed boundary can have hundreds of vertices, and SDA has a practical limit on how much text it will accept in one request. The fix is to simplify the polygon first: keep its overall shape, but reduce the number of points that define its edge.


In [ ]:
# Get the exact watershed boundary geometry
watershed_geom = target_wgs84.geometry.iloc[0]

# Convert to WKT without simplifying, just to see how large it is
raw_wkt = watershed_geom.wkt
print(f"Raw watershed WKT length: {len(raw_wkt):,} characters")

# Simplify the polygon
#
# The tolerance (0.001 degrees, roughly 100 meters) controls how much
# detail is removed. A larger tolerance means fewer vertices.
watershed_geom_simplified = watershed_geom.simplify(0.001)
simplified_wkt = watershed_geom_simplified.wkt
print(f"Simplified watershed WKT length: {len(simplified_wkt):,} characters")


### Step 2: Find Every Map Unit That Touches the Watershed


In [ ]:
watershed_mukey_sql = f"""
SELECT DISTINCT mukey
FROM SDA_Get_Mukey_from_intersection_with_WktWgs84('{simplified_wkt}')
"""

watershed_mukeys_df = query_sda(watershed_mukey_sql)
watershed_mukeys = watershed_mukeys_df['mukey'].tolist()

print(f"Found {len(watershed_mukeys)} map units intersecting the watershed:")
print(watershed_mukeys)


### Step 3: Retrieve Component and Horizon Data for All of Them

This is the same join pattern as Part 5, but for every map unit in the watershed at once, using `IN (...)` instead of a single `mukey =`. We also limit horizons to the top 100 cm, since that is the depth range most relevant to infiltration and near-surface runoff behavior.


In [ ]:
mukey_list_sql = ",".join(watershed_mukeys)

watershed_detail_sql = f"""
SELECT
    mu.mukey,
    mu.musym,
    mu.muname,
    co.cokey,
    co.compname,
    co.comppct_r,
    co.hydgrp,
    ch.hzdept_r,
    ch.hzdepb_r,
    ch.ksat_r
FROM mapunit mu
INNER JOIN component co ON mu.mukey = co.mukey
INNER JOIN chorizon ch ON co.cokey = ch.cokey
WHERE mu.mukey IN ({mukey_list_sql})
  AND ch.hzdept_r < 100
ORDER BY mu.mukey, co.comppct_r DESC, ch.hzdept_r ASC
"""

watershed_detail = query_sda(watershed_detail_sql)

numeric_cols = ['comppct_r', 'hzdept_r', 'hzdepb_r', 'ksat_r']
for col in numeric_cols:
    watershed_detail[col] = pd.to_numeric(watershed_detail[col], errors='coerce')

print(f"Retrieved {len(watershed_detail)} horizon records across {watershed_detail['mukey'].nunique()} map units")
watershed_detail.head(10)


### Step 4: Depth-Weighted Ksat, One Component at a Time

A single component can have several horizons at different depths, each with its own Ksat. To get one Ksat value per component, we weight by how thick each horizon is within our 0 to 100 cm window. A horizon that is 40 cm thick should count more than one that is only 5 cm thick.

This is done in clear, separate steps rather than one dense line of code, so each part of the calculation is visible.


In [ ]:
# Cap every horizon's bottom depth at 100 cm, since deeper material
# is outside our analysis window
watershed_detail['hzdepb_capped'] = watershed_detail['hzdepb_r'].clip(upper=100)

# Thickness of each horizon within the 0 to 100 cm window
watershed_detail['thickness'] = (
    watershed_detail['hzdepb_capped'] - watershed_detail['hzdept_r']
).clip(lower=0)

# Multiply each horizon's Ksat by its thickness
# This lets us compute a thickness-weighted average in the next step
watershed_detail['ksat_x_thickness'] = (
    watershed_detail['ksat_r'] * watershed_detail['thickness']
)

# Some horizons have no Ksat value on file. If we left their thickness
# in the denominator, the average would be pulled down as if those
# horizons had Ksat = 0, which is not true; they are simply unknown.
# So the denominator only counts thickness where Ksat is present.
watershed_detail['thickness_with_ksat'] = watershed_detail['thickness'].where(
    watershed_detail['ksat_r'].notna(), 0
)

# Group by component and sum
component_sums = watershed_detail.groupby(
    ['mukey', 'cokey', 'compname', 'comppct_r'], as_index=False
)[['ksat_x_thickness', 'thickness_with_ksat']].sum()

# Depth-weighted Ksat per component:
# total (ksat x thickness) divided by total thickness that has Ksat data
component_sums['comp_ksat'] = (
    component_sums['ksat_x_thickness'] / component_sums['thickness_with_ksat']
)

print(f"Depth-weighted Ksat calculated for {len(component_sums)} components")
component_sums[['mukey', 'compname', 'comppct_r', 'thickness_with_ksat', 'comp_ksat']].head(10)

### Step 5: Component-Weighted Ksat, One Map Unit at a Time

Now we combine components within each map unit, weighted by `comppct_r`. As flagged in Part 5, the percentages present after our inner join may not sum to 100. We handle this by dividing by the total percentage actually represented, and we report that total so nothing is hidden.


In [ ]:
# A component whose horizons all lack Ksat data ends up with a missing
# comp_ksat. Drop those before weighting, so they do not distort the
# average, and so pct_represented honestly reports how much of the map
# unit's area actually contributed to the estimate.
components_with_ksat = component_sums.dropna(subset=['comp_ksat'])

# Multiply each component's Ksat by its percentage of the map unit
components_with_ksat = components_with_ksat.copy()
components_with_ksat['ksat_x_pct'] = (
    components_with_ksat['comp_ksat'] * components_with_ksat['comppct_r']
)

# Group by map unit and sum both columns
mapunit_ksat = components_with_ksat.groupby('mukey', as_index=False)[
    ['ksat_x_pct', 'comppct_r']
].sum()

# Component-weighted Ksat per map unit
mapunit_ksat['ksat_umsec'] = (
    mapunit_ksat['ksat_x_pct'] / mapunit_ksat['comppct_r']
)

# Rename for clarity: this column tracks how much of the map unit's
# area is actually represented in our Ksat estimate
mapunit_ksat = mapunit_ksat.rename(columns={'comppct_r': 'pct_represented'})

# Convert to inches per hour for engineering use
mapunit_ksat['ksat_in_hr'] = mapunit_ksat['ksat_umsec'] * 0.14173

print(mapunit_ksat[['mukey', 'ksat_umsec', 'ksat_in_hr', 'pct_represented']])

### What This Output Tells Us

Look closely at the `pct_represented` column. Some map units show close to 100, meaning nearly the full mapped area is reflected in the Ksat estimate. Others show much lower values, in some cases as low as 25 to 30 percent. Those low numbers mean most of that map unit's area is made up of components with no horizon data on file within our depth window, often because they are water, rock outcrop, or another feature without a standard soil profile.

This is exactly the kind of caveat the method in Part 1 asked you to carry forward: never report a Ksat value without also stating how confident you are in the aggregation behind it. A map unit with `pct_represented` near 100 supports a confident Ksat estimate. One with `pct_represented` near 25 does not, and should be flagged for review before it goes into a model.

### Step 6: Dominant-Condition Hydrologic Soil Group

For hydrologic soil group, NRCS provides a convenience table called `muaggatt` that has already computed the dominant condition for you at the map unit level, so you do not need to do component weighting by hand for this variable.


In [ ]:
hsg_sql = f"""
SELECT mukey, musym, muname, hydgrpdcd, drclassdcd, aws0100wta
FROM muaggatt
WHERE mukey IN ({mukey_list_sql})
"""

hsg_df = query_sda(hsg_sql)
hsg_df['aws0100wta'] = pd.to_numeric(hsg_df['aws0100wta'], errors='coerce')

print(hsg_df[['mukey', 'muname', 'hydgrpdcd', 'drclassdcd', 'aws0100wta']])


Notice a few map units show `None` for `hydgrpdcd`. This happens for map units without a standard soil profile, such as open water, and is expected. Leave these as missing values rather than guessing a hydrologic soil group for them; a downstream curve number calculation should treat them as a distinct land cover / water case, not silently fill in a soil-based value.

🤖 **Try asking your AI assistant:** *"I extracted Ksat values from SSURGO for a watershed. Explain how these values should and should not be used in HEC-HMS model parameterization. Include cautions about component weighting, depth intervals, and calibration."*


## Part 7: Visualization - Mapping Hydrologic Soil Groups 📊

### Getting Map Unit Geometry

So far we have worked entirely with tables. To make a map, we need the actual polygon shapes of the map units, not just their keys. SDA provides these through a separate spatial web service (WFS, Web Feature Service) that returns geometry for a bounding box.

This is a different service than the one used earlier, so its response comes back in a different format: GML (Geography Markup Language) instead of JSON. GeoPandas can read GML directly.


In [ ]:
# Get the bounding box of our watershed in WGS84
minx, miny, maxx, maxy = target_wgs84.total_bounds
bbox_str = f"{minx},{miny},{maxx},{maxy}"

wfs_url = "https://sdmdataaccess.sc.egov.usda.gov/Spatial/SDMWGS84Geographic.wfs"
wfs_params = {
    "SERVICE": "WFS",
    "VERSION": "1.1.0",
    "REQUEST": "GetFeature",
    "TYPENAME": "MapunitPoly",
    "BBOX": bbox_str,
    "SRSNAME": "EPSG:4326",
}

wfs_response = requests.get(wfs_url, params=wfs_params, timeout=60)
print(f"Status code: {wfs_response.status_code}")
print(f"Response size: {len(wfs_response.content):,} bytes")

# Save the response to a file, since GeoPandas reads GML from a file path
with open("mapunit_polygons.gml", "wb") as f:
    f.write(wfs_response.content)

mupoly = gpd.read_file("mapunit_polygons.gml")
print(f"\nLoaded {len(mupoly)} map unit polygon features")
print(f"Columns: {list(mupoly.columns)}")


### An Important CRS Gotcha: Axis Order

Before using this geometry, check whether it actually lines up with our watershed. Geographic coordinate systems like EPSG:4326 officially define their axis order as (latitude, longitude), but most GIS software, including GeoPandas, expects geometry stored as (longitude, latitude) instead, matching a standard (x, y) convention. This particular web service follows the official axis order, not the GeoPandas convention, so the coordinates come in backward and need to be swapped before anything will line up correctly.

We can confirm this by checking the bounds before making any assumption.


In [ ]:
mupoly_unset = mupoly.set_crs(4326)
print("Bounds before any correction:")
print(mupoly_unset.total_bounds)
print()
print("Our watershed's actual bounds, for comparison:")
print(target_wgs84.total_bounds)


The first two numbers should be longitude and latitude, roughly matching our watershed's bounds (longitude near -105, latitude near 41). Instead, the values are swapped: the first number is near 41 and the second is near -105. This confirms the axis order problem described above. The fix is to swap x and y in every geometry before setting the CRS.


In [ ]:
# Swap the x and y coordinates in every geometry
mupoly['geometry'] = mupoly['geometry'].apply(
    lambda geom: shapely.ops.transform(lambda x, y: (y, x), geom)
)

# Now it is safe to set the CRS
mupoly = mupoly.set_crs(4326)

print("Bounds after the swap:")
print(mupoly.total_bounds)


The bounds should now be close to our watershed's bounds. This kind of coordinate mismatch is common when working with different web services and different software, and checking bounds against something you already trust, like our watershed layer, is a reliable way to catch it early.

### Clipping and Preparing the Map

The bounding box query returned every map unit polygon touching a rectangle around our watershed, which is a larger area than the watershed itself. We clip to the exact watershed boundary, then merge multi-part polygons that share the same map unit.


In [ ]:
# Clip map unit polygons to the exact watershed boundary
mupoly_clipped = gpd.clip(mupoly, target_wgs84)

# A single map unit can appear as several separate polygon pieces
# ("islands") within the watershed. Dissolve merges them into one
# shape per map unit, which is what we want for a clean map.
mupoly_dissolved = mupoly_clipped.dissolve(by='mukey').reset_index()

print(f"Clipped to {len(mupoly_dissolved)} map units within the watershed")

# Join the dominant hydrologic soil group we retrieved in Part 6
mupoly_dissolved['mukey'] = mupoly_dissolved['mukey'].astype(str)
hsg_df['mukey'] = hsg_df['mukey'].astype(str)

mupoly_hsg = mupoly_dissolved.merge(
    hsg_df[['mukey', 'hydgrpdcd']], on='mukey', how='left'
)

print(mupoly_hsg[['mukey', 'hydgrpdcd']])


In [ ]:
# Plot the watershed's map units, colored by hydrologic soil group
fig, ax = plt.subplots(figsize=(9, 8))

mupoly_hsg.plot(
    column='hydgrpdcd',
    categorical=True,
    legend=True,
    edgecolor='black',
    linewidth=0.3,
    ax=ax,
    missing_kwds={'color': 'lightgrey', 'label': 'No data'},
)

target_wgs84.boundary.plot(ax=ax, color='black', linewidth=2)

ax.set_title('Dominant Hydrologic Soil Group\nHUC-12 101900090108, Town of South Greeley', fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()


### What This Map Tells Us

The watershed is dominated by a mix of hydrologic soil groups, with group B soils covering much of the area (moderate infiltration, moderate runoff potential) and areas of group D and areas with no data corresponding to the map units with poor horizon coverage identified back in Part 6. For a curve number calculation, this means the watershed cannot be treated as a single uniform soil group. A proper curve number workflow, combining this SSURGO output with the land cover data from Module 4, would compute a separate curve number for each combination of land cover and hydrologic soil group, then area-weight the results.


## Part 8: Export - Model-Ready Output Tables 📂

### Building Traceable Summary Tables

An engineering deliverable should always be traceable back to its source. Every table exported here includes the source, the access date, and the aggregation method used, so a reviewer can understand exactly how each number was produced.


In [ ]:
import datetime

access_date = datetime.date.today().isoformat()

# Point summary table
point_summary = point_detail.copy()
point_summary['mukey_at_point'] = point_mukey
point_summary['point_lon'] = point_lon
point_summary['point_lat'] = point_lat
point_summary['source'] = 'NRCS Soil Data Access'
point_summary['access_date'] = access_date

point_summary.to_csv('point_soil_summary.csv', index=False)
print(f"Saved point_soil_summary.csv ({len(point_summary)} rows)")

# Watershed summary table: one row per map unit
watershed_summary = mapunit_ksat.merge(
    hsg_df[['mukey', 'muname', 'hydgrpdcd', 'drclassdcd', 'aws0100wta']],
    on='mukey', how='left'
)
watershed_summary['huc12'] = target_huc
watershed_summary['aggregation_method'] = 'component-percentage and depth (0-100 cm) weighted'
watershed_summary['source'] = 'NRCS Soil Data Access'
watershed_summary['access_date'] = access_date

watershed_summary.to_csv('watershed_soil_summary.csv', index=False)
print(f"Saved watershed_soil_summary.csv ({len(watershed_summary)} rows)")

watershed_summary


### Exporting the SSURGO Map Units as GIS Layers

The tables above are ready for a report, but a GIS user on your project team will also want the actual soil polygons for the watershed. We already built them: `mupoly_hsg` holds the map unit polygons clipped to the AOI, and we can attach the Ksat summary before saving.

Two formats are worth knowing:

- **GeoPackage** (`.gpkg`) is the modern single-file format. Column names of any length, no companion files, opens directly in QGIS and ArcGIS Pro. Prefer it when you have a choice.
- **Shapefile** (`.shp`) is the older industry standard many teams still request. It has two quirks you saw in Module 3: it is really a family of files that must travel together (so we ZIP it), and column names are limited to 10 characters, so longer names like `pct_represented` get truncated on save. We rename columns to short versions first so the truncation is deliberate instead of accidental.


In [ ]:
# Attach the Ksat summary to the clipped polygons, so the GIS layer
# carries the engineering attributes, not just shapes
mupoly_export = mupoly_hsg.merge(
    mapunit_ksat[['mukey', 'ksat_umsec', 'ksat_in_hr', 'pct_represented']],
    on='mukey', how='left'
)

# Keep only the useful columns
mupoly_export = mupoly_export[[
    'mukey', 'musym', 'hydgrpdcd', 'ksat_umsec', 'ksat_in_hr',
    'pct_represented', 'geometry'
]]

# Export 1: GeoPackage, the recommended modern format
mupoly_export.to_file('ssurgo_mapunits_aoi.gpkg', driver='GPKG')
print("Saved ssurgo_mapunits_aoi.gpkg")

# Export 2: Shapefile, for teams that request the older standard
#
# Shapefile column names are limited to 10 characters, so rename the
# long ones ourselves rather than letting the format truncate them
shp_export = mupoly_export.rename(columns={
    'hydgrpdcd': 'hsg_dom',
    'ksat_umsec': 'ksat_umsec',
    'ksat_in_hr': 'ksat_inhr',
    'pct_represented': 'pct_repr',
})

import os
import zipfile

os.makedirs('ssurgo_shp', exist_ok=True)
shp_export.to_file('ssurgo_shp/ssurgo_mapunits_aoi.shp')

# A shapefile is several files that must stay together, so ZIP the folder
with zipfile.ZipFile('ssurgo_mapunits_aoi_shp.zip', 'w') as zf:
    for fname in os.listdir('ssurgo_shp'):
        zf.write(os.path.join('ssurgo_shp', fname), fname)

print("Saved ssurgo_mapunits_aoi_shp.zip")
print("\nContents of the shapefile ZIP:")
for fname in sorted(os.listdir('ssurgo_shp')):
    print(f"   - {fname}")

In [ ]:
# Download all output files to your computer
from google.colab import files

files.download('point_soil_summary.csv')
files.download('watershed_soil_summary.csv')
files.download('ssurgo_mapunits_aoi.gpkg')
files.download('ssurgo_mapunits_aoi_shp.zip')

## Part 9: Beyond the API - SSURGO Portal and gNATSGO 🔭

This module used the SDA API because it fits a Colab-based, no-install workflow, and it doubles as an introduction to how web APIs work in general. Two other official NRCS tools are worth knowing about, even though this notebook does not use them directly.

### SSURGO Portal

SSURGO Portal is a free NRCS desktop application, not a web service. Given standard SSURGO download packages from Web Soil Survey, it builds a local GeoPackage or SQLite database, generates map unit raster GeoTIFFs, and produces Soil Data Viewer property tables that join directly to the map unit polygons or rasters by `mukey`.

Reach for SSURGO Portal instead of the API when a project needs:

- An offline, reproducible local database for repeated use
- Map unit raster GeoTIFFs for gridded modeling
- A workflow that opens cleanly in ArcGIS Pro, QGIS, or DB Browser for SQLite

This module does not use SSURGO Portal because it is a desktop application, and this course runs entirely in Google Colab. The workflow this module taught, mapunit joined to component joined to horizon, is the same relational structure SSURGO Portal's database uses internally, so the concepts transfer directly if you later work with it.

### gNATSGO

The gNATSGO Soil Database is a composite product: SSURGO filled in with STATSGO2 (a coarser, older survey) and Raster Soil Survey data wherever SSURGO has gaps, to provide seamless coverage across the entire country. It is distributed as a ready-to-use raster, which makes it well suited to large-area or national-scale work, and it is accessible through cloud platforms such as Microsoft's Planetary Computer.

gNATSGO is not raw SSURGO. Any time you use it, note in your documentation that some of the values may come from the coarser STATSGO2 source rather than survey-scale SSURGO mapping. For a single HUC-12 watershed like the one in this module, SSURGO through SDA is the more precise and more appropriate choice. gNATSGO becomes the better tool once a project spans multiple counties or states and downloading individual SSURGO surveys becomes impractical.

### Choosing Among the Three

| If you need... | Use |
|---|---|
| A quick, project-specific query for a point or watershed | Soil Data Access (this module) |
| An offline database, map unit rasters, Soil Data Viewer outputs | SSURGO Portal |
| Seamless national or multi-state raster coverage | gNATSGO |


## Engineering Cautions ⚠️

1. **Always state your aggregation method.** A SSURGO-derived value is never just "the Ksat" or "the hydrologic soil group." It is the dominant component, or a percentage-weighted average, or a depth-weighted average over a specific interval. Say which one, every time.

2. **Ksat is a starting point, not a calibrated parameter.** Use it to set up a model, then calibrate against observed streamflow or infiltration testing wherever that data exists.

3. **Watch `pct_represented`.** When an inner join drops minor components without horizon data, the percentage remaining may be well under 100. A low value means the reported Ksat reflects only a minority of that map unit's area, and should be flagged before use.

4. **SSURGO is survey-scale, not site-specific.** Treat every value in this module as screening-level. It supports early model setup and watershed-wide comparisons, not final design without site investigation.

5. **Units matter.** Ksat comes back in micrometers per second. Horizon depths come back in centimeters. Convert deliberately, and label every column with its units.

6. **The API can be down or slow.** NRCS services occasionally return errors or time out. This notebook's `query_sda()` function checks the status code and prints a clear message if something goes wrong; if a live query fails, use the fallback CSVs in the `data/` folder to keep working.


## Troubleshooting

| Problem | Likely Cause | What to Try |
|---|---|---|
| Query returns an empty table | Point or watershed falls outside a mapped SSURGO area, or a table/column name is misspelled | Print the exact SQL string and check it manually; confirm the location has SSURGO coverage |
| Point lands in the wrong place | WKT coordinate order reversed | WKT points are `POINT(longitude latitude)`, not `POINT(latitude longitude)` |
| Watershed query fails or times out | Polygon WKT is too large | Increase the simplification tolerance in `geometry.simplify()` |
| Map unit percentages do not sum to 100 | Minor components without horizon data were dropped by an inner join | Expected behavior; report `pct_represented` alongside any weighted value |
| Map does not line up with the watershed boundary | WFS response axis order is (latitude, longitude), not (longitude, latitude) | Swap x and y before setting the CRS, as shown in Part 7 |
| API request returns a non-200 status code | Network issue, or a temporary NRCS service outage | Print `response.status_code` and `response.text`; retry, or use the fallback CSVs |


## Practice Exercises 🎯

Each exercise below can be completed by lightly editing code that already appears in this notebook. Write your code in the cell under each description, run it, and check the output.

### Exercise 1: Query Your Own Point

Change the point coordinates in Part 5 to any location you know (your home, a project site, a landmark) and rerun the point query. You only need to change the `point_lon` and `point_lat` values.


In [ ]:
# EXERCISE 1: Query your own point
# Your code here.
#
# Steps:
# 1. Set point_lon and point_lat to a location you know.
# 2. Build the WKT string: POINT(longitude latitude).
# 3. Run the SDA_Get_Mukey_from_intersection_with_WktWgs84 query.
# 4. Print the mukey that comes back.


### Exercise 2: Add One More Variable

The Part 5 query already selects `hydgrp` and `drainagecl`. Add `taxorder` (soil taxonomic order, from the `component` table) to the `SELECT` list and rerun the query. This is a one-line change to a query that is already written for you.


In [ ]:
# EXERCISE 2: Add one more variable
# Your code here.
#
# Steps:
# 1. Copy the point_detail_sql query from Part 5.
# 2. Add co.taxorder to the SELECT list.
# 3. Run it with query_sda() and print the result.


### Exercise 3: Change the Depth Window

Part 6 calculates a depth-weighted Ksat using horizons above 100 cm. Change this to 50 cm instead, rerun the weighting steps, and compare the watershed-average Ksat for both depth windows in a small two-row table.


In [ ]:
# EXERCISE 3: Change the depth window
# Your code here.
#
# Steps:
# 1. Copy the Part 6 depth-weighting code (Steps 4 and 5).
# 2. Change every 100 to 50 (the hzdept_r filter, the clip upper bound).
# 3. Compute the new watershed-average Ksat.
# 4. Print a small table comparing the 100 cm and 50 cm results.


### Challenge Exercise: A Reusable Point-Query Function (AI-Assisted)

Ask your AI assistant to help you wrap the Part 5 point-query workflow into a single function, `get_soil_at_point(lat, lon)`, that returns a small summary DataFrame for any location. Test it on two different points.

Suggested prompt to paste into your AI assistant:

*"Help me write a Python function called get_soil_at_point(lat, lon) that uses the query_sda function already defined in my notebook to find the mukey at a point, then retrieves comppct_r, hydgrp, and a depth-weighted Ksat for that map unit, and returns it as a one-row DataFrame."*


In [ ]:
# CHALLENGE EXERCISE: Build get_soil_at_point(lat, lon)
# Your code here.
#
# Steps:
# 1. Ask your AI assistant using the suggested prompt above.
# 2. Review the function it suggests. Does it reuse query_sda()?
#    Does it handle an empty result?
# 3. Test the function on two points and print both results.


## That's Module 7 Done!

Think about what this module automated. Finding the right soil survey area, looking up map units, joining components and horizons by hand, and computing a defensible watershed-average Ksat used to mean hours in Web Soil Survey and a spreadsheet. Here, it runs in a single script, and you can point it at any watershed in the country by changing one shapefile.

### What you can do now

- **Explain SSURGO's structure.** Map unit, component, and horizon, connected by `mukey`, `cokey`, and `chkey`.
- **Query SDA directly.** Retrieve soil properties for a point or a watershed using raw web requests, not a wrapper library.
- **Aggregate correctly.** Combine horizons by depth and components by percentage, while tracking how much of each map unit's area is actually represented.
- **Produce model-ready outputs.** Export traceable, unit-labeled summary tables ready for a curve number or infiltration workflow.

### Next Steps

- **Combine with Module 4.** Curve numbers need both land cover and hydrologic soil group. Module 4 covered the land cover half; this module covered the soil half.
- **On your own.** Run this workflow on a watershed from a current project, and compare the Ksat and hydrologic soil group results against any calibration data you already have.
- **When a project outgrows one watershed.** If you need seamless coverage across many counties or states, revisit Part 9 and consider gNATSGO instead of looping this workflow over many SSURGO queries.

The full repository, including data files and the rest of the course, is at [github.com/mohsennasab/python-fundamentals-hh](https://github.com/mohsennasab/python-fundamentals-hh).
